In [ ]:
# !pip install boto3

In [20]:
import numpy as np
import pandas as pd
import boto3
import os
from dotenv import load_dotenv
from io import StringIO
from gensim.models import Word2Vec
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

In [21]:
load_dotenv()
s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

bucket_name = os.getenv("S3_BUCKET_NAME")
key = "processed/data_cleaned.csv"

s3.download_file(bucket_name, key, "../data/processed/data_cleaned.csv")

In [22]:
data = pd.read_csv("../data/processed/data_cleaned.csv")
data.head()

,id,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [23]:
X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [24]:
def tokenize(text):
    return text.lower().split()

sentences = [tokenize(text) for text in X_train]
print(sentences[0])

['president', 'obama', 'offered', 'enthusiastic', 'support', 'for', 'hillary', 'clinton', 'at', 'the', 'democratic', 'national', 'convention', 'wednesday', 'as', 'he', 'painted', 'a', 'hopeful', 'picture', 'of', 'the', 'country.', 'u.s.', 'president', 'barack', 'obama', '(l)', 'and', 'democratic', 'presidential', 'candidate', 'hillary', 'clinton', 'wave', 'to', 'the', 'crowd', 'after', 'the', 'president', 'spoke', 'at', 'the', 'democratic', 'national', 'convention', 'in', 'philadelphia,', 'penn.,', 'on', 'july', '27,', '2016.', 'president', 'obama', 'described', 'an', 'optimistic,', 'hopeful', 'picture', 'of', 'america', 'in', 'a', 'speech', 'wednesday', 'night', 'at', 'the', 'democratic', 'national', 'convention,', 'pointedly', 'diverging', 'from', 'the', 'more', 'foreboding', 'tone', 'of', 'the', 'previous', "week's", 'republican', 'event.', 'mr.', 'obama', 'offered', 'an', 'enthusiastic', 'endorsement', 'of', 'hillary', 'clinton,', 'saying', '“nobody', '[is]', 'more', 'qualified”', 

In [ ]:
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2)

In [ ]:
class Word2VecTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model):
        self.model = model
        self.dim = model.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        vectors = []
        for text in X:
            words = tokenize(text)
            vecs = [self.model.wv[w] for w in words if w in self.model.wv]
            vectors.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.dim))
        return np.array(vectors)

In [ ]:
model = Pipeline([
    ("w2v", Word2VecTransformer(w2v_model)),
    ("clf", LinearSVC(class_weight='balanced'))
])

model.fit(X_train, y_train)

c:\Users\nada amalia\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Pipeline(steps=[('w2v',
                 Word2VecTransformer(model=<gensim.models.word2vec.Word2Vec object at 0x00000230458BD340>)),
                ('clf', LinearSVC(class_weight='balanced'))])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
def predict_text(text_input, model):
    text_input = str(text_input)
    pred = model.predict([text_input])[0]
    return "Fake" if pred == 1 else "Real"


user_input = input("Input text: ")
hasil = predict_text(user_input, model)

print("Input:", user_input)
print("Label:", hasil)

In [ ]:
# !pip install joblib

In [ ]:
import joblib
joblib.dump(model, "../outputs/model.pkl")

['../outputs/model.pkl']